# Cell 1 — PredMain AI Demo

End-to-end predictive maintenance walkthrough for data loading, preprocessing, model training, evaluation, plotting, and CSV inference.

In [ ]:
# Cell 2 — Install dependencies
!pip install -r ../requirements.txt

In [ ]:
# Cell 3 — Load and preview raw data
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_loader import create_failure_target, load_ai4i_dataset
from models import predict_from_csv, run_training
from evaluate import evaluate_models

raw_df = load_ai4i_dataset()
raw_df = create_failure_target(raw_df)
display(raw_df.head())
raw_df['failure'].value_counts(normalize=True)

In [ ]:
# Cell 4 — Run preprocessing, show split summary
from data_loader import run_preprocessing

processed_splits = run_preprocessing()
{name: split.shape for name, split in processed_splits.items()}

In [ ]:
# Cell 5 — Train Isolation Forest, print results
training_results = run_training()
training_results['models']['Isolation Forest']

In [ ]:
# Cell 6 — Train Autoencoder, show loss curve inline
from IPython.display import Image, display

display(Image(filename=str(PROJECT_ROOT / 'outputs' / 'plots' / 'autoencoder_loss.png')))
training_results['models']['Autoencoder']

In [ ]:
# Cell 7 — Train LSTM, show epoch progress inline
training_results['models']['LSTM']['history'][:5]

In [ ]:
# Cell 8 — Run evaluation, display leaderboard table
evaluation_results = evaluate_models()
display(evaluation_results['evaluation']['comparison_table'])

In [ ]:
# Cell 9 — Display all plots inline
plot_names = [
    'feature_distributions.png',
    'correlation_heatmap.png',
    'timeseries_failures.png',
    'anomaly_score_distributions.png',
    'f1_comparison.png',
    'roc_comparison.png',
    'confusion_matrix.png',
    'autoencoder_loss.png',
]
for plot_name in plot_names:
    plot_path = PROJECT_ROOT / 'outputs' / 'plots' / plot_name
    if plot_path.exists():
        display(Image(filename=str(plot_path)))
    else:
        print(f'Missing plot: {plot_path}')

In [ ]:
# Cell 10 — CSV Upload Demo
# 🔍 TEST YOUR OWN DATA
# Point this at any CSV with the same sensor columns
# and get back anomaly predictions from all three models

csv_path = PROJECT_ROOT / 'data' / 'processed' / 'test_raw.csv'  # ← change this
results = predict_from_csv(csv_path)
display(results.head())

In [ ]:
# Cell 11 — Print executive summary
results_path = PROJECT_ROOT / 'outputs' / 'results.json'
payload = json.loads(results_path.read_text(encoding='utf-8'))
payload['evaluation']['comparison_table'][0]